# Hydra in Jupyter Notebooks (Compose API)

This notebook demonstrates how to use **Hydra** interactively within Jupyter Notebooks using the **Compose API**.

---

### Why Not `@hydra.main()` in Notebooks?
In standard Python scripts, Hydra uses the `@hydra.main()` decorator, which:
- Expects to parse command-line arguments (`sys.argv`).
- Controls program termination and directory changes.

Inside interactive environments like **Jupyter Notebooks**, **Google Colab**, or unit tests, there is no CLI entry point. Instead, Hydra provides the **Compose API** (`initialize`, `initialize_config_dir`, and `compose`) to dynamically compose configurations on-the-fly.


In [ ]:
# 1. Imports
import os
from hydra import initialize, initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf


## 2. Composing the Default Configuration
We use `with initialize(version_base=None, config_path="conf"):` as a **context manager**.
This ensures that Hydra initializes the config search path and cleanly releases its global state when the cell exits.


In [ ]:
with initialize(version_base=None, config_path="conf"):
    cfg = compose(config_name="config")
    print("=== Composed Default Configuration ===")
    print(OmegaConf.to_yaml(cfg))


## 3. Dynamic Interactive Overrides
You can simulate any command-line overrides directly within Python using the `overrides=[...]` list.
Below, we:
- Switch the database from `mysql` to `postgresql`
- Switch the model from `resnet` to `vit`
- Modify the learning rate to `0.0005`
- Append a new key `+optimizer=adamw`


In [ ]:
with initialize(version_base=None, config_path="conf"):
    cfg_override = compose(
        config_name="config",
        overrides=[
            "db=postgresql",
            "model=vit",
            "model.learning_rate=0.0005",
            "+optimizer=adamw"
        ]
    )
    print("=== Overridden Configuration ===")
    print(OmegaConf.to_yaml(cfg_override))


## 4. Accessing Config Fields in Python
The returned `cfg` object is an OmegaConf `DictConfig`, allowing both dictionary and attribute-style access.


In [ ]:
print(f"Experiment: {cfg_override.experiment_name}")
print(f"Database:   {cfg_override.db.name} on port {cfg_override.db.port}")
print(f"Model:      {cfg_override.model.name} (lr={cfg_override.model.learning_rate})")
print(f"Optimizer:  {cfg_override.optimizer}")


## 5. Using Absolute Directory Paths (`initialize_config_dir`)
If your configuration files are located elsewhere on the filesystem, use `initialize_config_dir` with an absolute path:


In [ ]:
abs_config_dir = os.path.abspath("conf")
print(f"Loading from absolute path: {abs_config_dir}")

with initialize_config_dir(version_base=None, config_dir=abs_config_dir):
    cfg_abs = compose(config_name="config", overrides=["debug=true", "db.port=3307"])
    print(f"Debug mode: {cfg_abs.debug}")
    print(f"DB Port:    {cfg_abs.db.port}")


## 6. Safe Re-execution Pattern (Handling GlobalHydra)
If you call `initialize()` outside a context manager, re-running the cell will throw:
`GlobalHydra is already initialized`.

To prevent this in production notebooks, always clear `GlobalHydra` before initializing:


In [ ]:
# Clear any existing Hydra global state
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

# Initialize globally
initialize(version_base=None, config_path="conf")
cfg_global = compose(config_name="config", overrides=["model.batch_size=64"])
print(f"Global Compose Batch Size: {cfg_global.model.batch_size}")

# Always clean up when finished
GlobalHydra.instance().clear()
print("GlobalHydra successfully cleared.")
